# 3 · Claude Code II: Debugging, Testing & the Earnings Engine

**You leave with:** an AI-generated investment-memo draft from your own Earnings Analysis Engine: with a verification layer that catches fabricated evidence automatically.

## The debugging protocol

1. **Read the traceback bottom-up**: last line says *what*, the marked line says *where*.
2. Reproduce it. 3. **Diagnose before fixing** (make Claude explain the cause first). 4. One change at a time.

> Crashes are the *friendly* bugs: they announce themselves. The dangerous ones return a number.

**Tests are financial logic written down.** The most important one is the *napkin test*: a case simple enough to compute by hand. If your model can't reproduce a hand-checkable case, you don't have a model: you have a rumour.

In [ ]:
import sys, os, json
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except ImportError:
    pass
HAS_KEY = bool(os.environ.get("ANTHROPIC_API_KEY"))
print(f"repo root: {ROOT}")
print(f"API key:   {'configured' if HAS_KEY else 'NOT SET - cells that call Claude will be skipped'}")

## Part A: watch a broken DCF fail (follow along)

`session-03-debugging/demo/broken_dcf.py` values **Meridian Semiconductor** (fictional; trades at $62). It ships with 6 planted bugs. Watch it crash:

In [ ]:
import subprocess
r = subprocess.run([sys.executable, str(ROOT / "session-03-debugging" / "demo" / "broken_dcf.py")],
                   capture_output=True, text=True)
print(r.stdout[-300:] if r.stdout else "", r.stderr[-500:])

In [ ]:
# and watch its sanity tests fail (7 failures = 7 pieces of violated financial logic)
r = subprocess.run([sys.executable, "-m", "pytest", str(ROOT / "session-03-debugging" / "demo"), "-q"],
                   capture_output=True, text=True)
print(r.stdout[-600:])

In class the instructor fixes that file live with Claude Code, test by test. Here **you build the correct model yourself**, function by function: each with its test right below. The specs are in the docstrings; the finance is in the comments.

## Part B: LAB 1: build the DCF right

### Exercise 1: WACC (don't forget the tax shield)

In [ ]:
def wacc(equity_weight, debt_weight, cost_of_equity, cost_of_debt, tax_rate):
    """WACC = w_e * k_e + w_d * k_d * (1 - tax_rate). Interest is tax-deductible:
    debt is cheaper than it looks."""
### START CODE HERE ###
    return equity_weight * None + debt_weight * None * (1 - None)   # don't forget the tax shield
### END CODE HERE ###

print(f"Meridian WACC: {wacc(0.85, 0.15, 0.115, 0.055, 0.21):.2%}")

In [ ]:
# ✅ self-check: run me
assert abs(wacc(0.0, 1.0, 0.10, 0.05, 0.21) - 0.05 * 0.79) < 1e-12, "100% debt at 5%, 21% tax -> 3.95%"
assert abs(wacc(1.0, 0.0, 0.115, 0.05, 0.21) - 0.115) < 1e-12
print("All checks passed ✅")

### Exercise 2: project cash flows (one growth rate per year, ANY horizon)

In [ ]:
def project_fcf(base_fcf, growth_rates):
    """project_fcf(100, [0.10, 0.10]) -> [110.0, 121.0]. Horizon = len(growth_rates),
    never hardcoded."""
### START CODE HERE ###
    flows, fcf = [], base_fcf
    for g in growth_rates:
        fcf = fcf * (1 + None)
        flows.append(None)
    return flows
### END CODE HERE ###

project_fcf(1.35, [0.30, 0.25, 0.20, 0.15, 0.10])

In [ ]:
# ✅ self-check: run me
f = project_fcf(100.0, [0.10, 0.10, 0.10])
assert len(f) == 3, "3 rates in -> 3 flows out (the broken model hardcoded 5 and crashed)"
assert abs(f[2] - 133.1) < 1e-9
print("All checks passed ✅")

### Exercise 3: terminal value (and refuse impossible inputs)

`TV = FCF_final × (1+g) / (r − g)`. If `g >= r` the formula implies infinite value: **raise `ValueError`** instead of dividing. Validation is part of the model.

In [ ]:
def terminal_value(final_fcf, terminal_growth, discount_rate):
### START CODE HERE ###
    if None >= None:                       # which comparison implies infinite value?
        raise ValueError("terminal growth must be below the discount rate - otherwise value is infinite")
    return final_fcf * (1 + None) / (None - None)
### END CODE HERE ###

print(f"TV example: {terminal_value(100, 0.02, 0.10):,.0f}")

In [ ]:
# ✅ self-check: run me
assert abs(terminal_value(100, 0.0, 0.10) - 1000.0) < 1e-9
try:
    terminal_value(100, 0.12, 0.10)
    raise AssertionError("g >= r must raise ValueError, not return a number")
except ValueError:
    pass
print("All checks passed ✅")

### Exercise 4: the full DCF (discounting is where the bugs hide)

Year *t* discounted at `(1+r)**t` with **t=1 for the first year** (money in a year is worth less NOW). TV sits at the END of year N: discount it by `(1+r)**N`. Equity = EV **minus** net debt (debt holders get paid first).

In [ ]:
def dcf_value(base_fcf, growth_rates, discount_rate, terminal_growth, net_debt, shares_outstanding):
### START CODE HERE ###
    flows = project_fcf(base_fcf, growth_rates)
    n = len(flows)
    pv_explicit = sum(f / (1 + discount_rate) ** t
                      for t, f in enumerate(flows, start=None))   # the FIRST year is t = ?
    pv_terminal = terminal_value(flows[-1], terminal_growth, discount_rate) / (1 + discount_rate) ** None
    enterprise_value = None + None
    equity_value = enterprise_value - None                        # who gets paid first?
    per_share = equity_value / None
### END CODE HERE ###
    return {"enterprise_value": enterprise_value, "equity_value": equity_value,
            "per_share": per_share, "pv_explicit": pv_explicit, "pv_terminal": pv_terminal}

In [ ]:
# ✅ self-check: the NAPKIN TEST: flat FCF 100, r=10%, g=0, 2 years.
# PV explicit = 100/1.1 + 100/1.21 = 173.55; TV = 1000, PV(TV) = 826.45; EV = 1000.00 exactly.
r = dcf_value(100.0, [0.0, 0.0], 0.10, 0.0, net_debt=200.0, shares_outstanding=10.0)
assert abs(r["enterprise_value"] - 1000.0) < 0.01, f"EV should be 1000.00, got {r['enterprise_value']:.2f}"
assert abs(r["equity_value"] - 800.0) < 0.01, "equity = EV - net debt (SUBTRACT)"
assert abs(r["per_share"] - 80.0) < 0.001
print("All checks passed ✅ - your model reproduces a hand-checkable case")

In [ ]:
# Now value Meridian for real:
rate = wacc(0.85, 0.15, 0.115, 0.055, 0.21)
result = dcf_value(1.35, [0.30, 0.25, 0.20, 0.15, 0.10], rate, 0.025,
                   net_debt=0.85, shares_outstanding=0.46)
print(f"WACC {rate:.2%} | EV ${result['enterprise_value']:.2f}bn | "
      f"per share ${result['per_share']:.2f}  (market: $62.00)")
assert abs(result["per_share"] - 75.61) < 0.05, "expected ~$75.61/share"
print("+22% upside - IF you believe the growth assumptions. Green tests AND a plausible number: you need both.")

## Part C: LAB 2: the Earnings Analysis Engine

From an earnings-call transcript to a structured, **evidence-verified** note. The transcript (`session-03-debugging/data/transcript_meridian_q2_fy2026.txt`) is synthetic and the company fictional: so the model can't lean on memorized knowledge: everything must come from the document.

The plumbing (schema + API call) is imported from the course solution; **your work is the trust layer**.

In [ ]:
sol_dir = ROOT / "session-03-debugging" / "solutions"
sys.path.insert(0, str(sol_dir))
from earnings_engine import EARNINGS_SCHEMA, analyze, render_memo, DEFAULT_TRANSCRIPT

transcript = DEFAULT_TRANSCRIPT.read_text()
print(f"{len(transcript.split())} words. Speakers: CEO, CFO, five analysts. Somewhere in here: 7 red flags.")

### Exercise 5: verify_evidence: the fabrication detector

Every claim the model makes carries a verbatim quote. You check each quote against the source **in plain Python**. Normalize both sides first (collapse whitespace, lowercase, straighten curly quotes) so formatting can't cause false negatives. Set `item["verified"]` on every item in `key_themes`, `risks`, `red_flags`, and store totals in `analysis["_verification"]`.

In [ ]:
import re

def _normalize(text: str) -> str:
    """Whitespace-collapse + casefold + straighten curly quotes. GIVEN - it's
    plumbing; YOUR work is the verification logic below."""
    text = text.replace("\u2019", "'").replace("\u2018", "'")
    text = text.replace("\u201c", '"').replace("\u201d", '"')
    return re.sub(r"\s+", " ", text).casefold().strip()

def verify_evidence(analysis: dict, transcript: str) -> dict:
### START CODE HERE ###
    haystack = _normalize(None)                        # normalize which text?
    checked = failed = 0
    for section in ("key_themes", "risks", "red_flags"):
        for item in analysis.get(section, []):
            quote = item.get("evidence_quote", "")
            item["verified"] = bool(quote) and None in haystack   # hint: the NORMALIZED quote
            checked += 1
            failed += 0 if item["verified"] else 1
    analysis["_verification"] = {"quotes_checked": None, "quotes_failed": None}
### END CODE HERE ###
    return analysis

print("defined - now catch a fabrication:")

In [ ]:
# ✅ self-check: run me. The canned dry-run analysis hides ONE deliberately
# fabricated quote. If your verify_evidence works, it catches exactly that one.
analysis = verify_evidence(analyze(transcript, dry_run=True), transcript)
v = analysis["_verification"]
print(f"quotes checked: {v['quotes_checked']}, failed: {v['quotes_failed']}")
assert v["quotes_checked"] >= 10, "check key_themes, risks AND red_flags"
assert v["quotes_failed"] == 1, "exactly ONE quote is fabricated - if 0, your matching is too loose; if >1, normalize better"
fake = [t for s in ("key_themes", "risks", "red_flags") for t in analysis[s] if not t["verified"]]
print("All checks passed ✅  Caught fabrication:", repr(fake[0]["evidence_quote"]))

That quote: a promised margin recovery: sounds completely plausible and appears nowhere in the transcript. **Your eyes would have missed it; your code didn't.** This is what 'verify, then trust' means in practice.

In [ ]:
# Render the full memo (uses the course renderer) and, if you have a key, run LIVE:
OUTD = ROOT / "outputs"; OUTD.mkdir(exist_ok=True)
(OUTD / "meridian_earnings_memo.md").write_text(render_memo(analysis, "transcript (dry-run)"))
print("outputs/meridian_earnings_memo.md written (dry-run).")

if HAS_KEY:
    live = verify_evidence(analyze(transcript, dry_run=False), transcript)
    lv = live["_verification"]
    (OUTD / "meridian_earnings_memo.md").write_text(render_memo(live, "transcript (LIVE)"))
    print(f"LIVE run: {lv['quotes_checked'] - lv['quotes_failed']}/{lv['quotes_checked']} quotes verified. Memo overwritten.")
    print("Did it find: recurring 'one-time' costs? the CEO/CFO margin gap? the guidance exclusion? DSO 71 vs 58?")
else:
    print("No API key - the dry-run memo still demonstrates the whole pipeline.")

## Deliverable checklist

- [ ] All DCF checks green, ending at **$75.61 vs $62 market**: and you can name each planted bug in one sentence (crash · year-1 discounting · tax shield · undiscounted TV · net-debt sign · missing g<r guard)
- [ ] Your `verify_evidence` catches **exactly 1** fabricated quote in dry-run
- [ ] `outputs/meridian_earnings_memo.md` generated (live if you have a key) and committed to your repo
- [ ] Stretch: numeric cross-check: regex the transcript for figures and confirm every number in the summary appears in the source

**Next:** `04-workflows-edgar.ipynb`: we stop pasting context and start FETCHING it: live SEC filings.